In [3]:
# Для CPU-bound задач (Много вычислений, мало ожиданий):
# Потоки (threading) почти бесполезны из-за GIL (Global Interpreter Lock)
# Нужно использовать процессы (multiprocessing) для реального параллелизма
# CPU-bound → используйте ProcessPoolExecutor
from concurrent.futures import ProcessPoolExecutor

In [4]:
# Для I/O-bound задач (Много ожиданий (сеть, диск)):
# Потоки (threading) очень эффективны, потому что GIL отпускается во время ожидания I/O
# Также отлично работают асинхронные подходы (async/await)
# I/O-bound → используйте ThreadPoolExecutor или asyncio
from concurrent.futures import ThreadPoolExecutor
# или
import asyncio

In [21]:
# Суть Asyncio
# asyncio — это кооперативная многозадачность:
# задачи добровольно отдают управление через await,
# и цикл событий умно распределяет время между ними.

In [12]:
# Это менеджер задач — бесконечный цикл, который:
# 1. Смотрит: какие задачи могут выполняться прямо сейчас?
# 2. Выполняет их немного (до первого await)
# 3. Если задача ожидает чего-то (сетевой ответ, таймер), она приостанавливается
# 4. Цикл переходит к следующей готовой задаче
# 5. Когда ожидаемое событие происходит (пришёл ответ от сервера), задача возобновляется
# Все проихсодит  в одном потоке

In [13]:
async def fetch_data():
# Создаётся сопрограмма (coroutine). Это не функция, а объект-план выполнения.
# Пока вы не вызвали её с await, ничего не происходит!
# Сопрограмма — это как "рецепт", который можно приостановить и возобновить

SyntaxError: incomplete input (4121629642.py, line 4)

In [15]:
await
# «Я сейчас запускаю эту сопрограмму, но если она начнёт ждать — дай управление обратно циклу событий, чтобы он мог делать другие дела».
# Когда доходит до await func(), функция приостанавливается
# Цикл событий переключается на другие задачи
# Как только func() получила необходимые данные — функция возобновляется с того места



# Когда вы вызываете await something, на самом деле происходит:
# coro = something()
# result = coro.send(None) или coro.throw(), coro.close()
# Метод .send() — это способ передать значение внутрь генератора и возобновить его выполнение
# await — это удобная обёртка над этой механикой (раньше использовали yield from)

SyntaxError: invalid syntax (1025483812.py, line 1)

In [16]:
# Откуда берётся "ожидание"?
# Цепочка await в итоге доходит до низкоуровневого объекта, который умеет сообщить циклу событий:
# «Я жду, пока сработает таймер / придёт пакет по сети / освободится диск».
# loop.sock_recv() — ждёт данные из сокета
# loop.call_later() — ждёт таймер
# asyncio.sleep() — ждёт время
# Эти объекты регистрируются в цикле событий, и когда нужное событие происходит — цикл возобновляет соответствующую сопрограмму.

In [17]:
# Почему это работает в одном потоке?
# Потому что все операции ввода-вывода неблокирующие:
# Вместо socket.recv() (который блокирует поток), используется loop.sock_recv(), который:
# Говорит ОС: «Сообщи мне, когда будут данные»
# Не ждёт, а сразу возвращает управление

# Цикл событий использует системные механизмы:
# epoll (Linux)
# kqueue (macOS)
# IOCP (Windows
# Эти механизмы позволяют одновременно следить за тысячами соединений без потоков


In [5]:
# Комбинация executor.submit и futures.as_completed обладает большей гибкостью, чем executor.map, потому что ей можно подавать различные вызываемые объекты и аргументы. 
# Тогда как executor.map предназначена для выполнения одного и того же вызываемого объекта с разными аргументами. 
# Кроме того, множество будущих объектов, передаваемых futures.as_completed, может поступать от нескольких исполнителей – 
# одни из них могли быть созданы экземпляром ThreadPoolExecutor, другие – экземпляром ProcessPoolExecutor.

In [2]:
# Платформенная сопрограмма
# Функция, определенная с помощью конструкции async def. Мы можем делегировать работу от одной платформенной сопрограммы другой,
#  воспользовавшись ключевым словом await, по аналогии с тем, как классические сопрограммы уступают управление с помощью предложения yield from. 
# Предложение async  def  всегда определяет платформенную сопрограмму, даже если в ее теле не встречается ключевое слово await. Слово await нельзя использовать вне платформенной сопрограммы

In [3]:
# Классическая сопрограмма
# Генераторная  функция,  которая  потребляет  данные,  отправленные  ей с помощью вызовов my_coro.send(data), и читает эти данные, используя yield в выражении. 
# Классическая сопрограмма может делегировать работу другой классической сопрограмме с помощью предложения yield from. 
# Классические сопрограммы не приводятся в действие словом await и более не поддерживаются библиотекой asyncio

In [4]:
# Генераторная сопрограмма
# Генераторная функция, снабженная декоратором @types.coroutine, включенным в Python 3.5. Этот декоратор делает генератор совместимым с новым ключевым словом await

In [5]:
# Асинхронный генератор
# Генераторная  функция,  определенная  с  помощью  конструкции  async  def и содержащая в теле yield. 
# Она возвращает асинхронный объект-генератор, предоставляющий метод __anext__ для асинхронного получения следующего элемента.

In [6]:
# Подробно про ключевые методы asyncio
# 1. asyncio.get_running_loop()
# Возвращает текущий цикл событий
# Безопасен: работает только внутри запущенного цикла
# Используется внутри сопрограмм

# 2. loop.getaddrinfo()
# Асинхронная обёртка над системным вызовом getaddrinfo()
# Не блокирует цикл событий во время DNS-запроса
# Аналоги: loop.sock_connect(), loop.run_in_executor() для других I/O-операций

# 3. asyncio.as_completed()
# Конкурентное выполнение: все сопрограммы запускаются одновременно
# Обработка по готовности: первые завершившиеся обрабатываются первыми
# Альтернативы:
# asyncio.gather() — ждёт все результаты, возвращает список
# asyncio.wait() — более низкоуровневый контроль

In [19]:
# Как запускать несколько задач одновременно?
# asyncio.create_task
# Обе задачи запущены параллельно в одном цикле.

task1 = asyncio.create_task(download_page("url1"))
task2 = asyncio.create_task(download_page("url2"))
await task1
await task2

NameError: name 'asyncio' is not defined

In [22]:
# asyncio.gather()
# Запускает все задачи и ждёт все результаты сразу.

async def task(name, delay):
    await asyncio.sleep(delay)
    return name

# Создаём сопрограммы
coros = [task('A', 2), task('B', 1), task('C', 3)]

# Запускаем конкурентно
results = await asyncio.gather(*coros) # Порядок сохранен, хотя B завершилось раньше всех





NameError: name 'asyncio' is not defined

In [2]:
# async with
# async with — это асинхронная версия контекстного менеджера.
# Он используется, когда вход или выход требуют асинхронных операций (например, сетевых вызовов).

# Требует два специальных метода:
# __aenter__() — асинхронный, вызывается при входе → должен возвращать ресурс
# __aexit__() — асинхронный, вызывается при выходе → может делать cleanup (закрытие соединений и т.п.)
# Оба метода — сопрограммы, поэтому их вызывают через await.

async with AsyncClient() as client:

# Класс httpx.AsyncClient реализует асинхронный контекстный менеджер:
# При входе в блок вызывается await client.__aenter__()
# 1. Инициализирует внутренние ресурсы (пулы соединений, event loop и т.д.)
# 2. Возвращает сам клиент (client)
# При выходе вызывается await client.__aexit__(...)
# 1. Корректно закрывает все HTTP-соединения
# 2. Освобождает ресурсы (очень важно для сетевых клиентов!)    

async def supervisor(cc_list: list[str]) -> int:
    async with AsyncClient() as client:          # ← Шаг 1: инициализация клиента
        to_do = [download_one(client, cc) for cc in sorted(cc_list)]  # ← Шаг 2: создание планов
        res = await asyncio.gather(*to_do)       # ← Шаг 3: конкурентное выполнение
    # ← Шаг 4: автоматическая очистка (закрытие соединений)
    return len(res)

# 1. Вход в async with
# → создаётся и настраивается HTTP-клиент с пулом соединений
# 2. Создание списка сопрограмм
# → каждая download_one(client, cc) получает один и тот же клиент
# → это эффективно: все запросы используют общий пул соединений
# 3. Конкурентное выполнение
# → gather запускает все загрузки параллельно
# → все они используют один клиент (безопасно в asyncio)
# 4. Выход из async with
# → автоматически вызывается __aexit__
# → все сетевые соединения закрываются корректно
# → даже если произошло исключение!

IndentationError: expected an indented block after 'with' statement on line 10 (292635001.py, line 20)

In [3]:
# asyncio.Semaphore
# Семафор — это примитив синхронизации, который ограничивает количество одновременно выполняющихся задач.

# В программировании:
# Семафор имеет счётчик (например, value=10)
# Каждая задача запрашивает разрешение (acquire)
# Если счётчик > 0 → задача проходит, счётчик уменьшается
# Если счётчик = 0 → задача ждет, пока кто-то освободит место (release)

import asyncio

semaphore = asyncio.Semaphore(10) # Максимум 10 одновременных задач

async def download_one():
    async with semaphore:          # ← запросить "место"
        image = await get_flag()  # ← только 10 таких вызовов одновременно
        save_flag()

# async with semaphore: вызывает await semaphore.acquire()
# Если есть свободные "слоты" — продолжает выполнение
# Если все слоты заняты — приостанавливает сопрограмму, пока не освободится место
# При выходе из блока автоматически вызывается semaphore.release()        

In [7]:
# Стандартные паттерны асинхронного кода


# Паттерн 1: Главный запуск через asyncio.run()
async def main():
    # ваш асинхронный код

if __name__ == '__main__':
    asyncio.run(main())

IndentationError: expected an indented block after function definition on line 5 (573992912.py, line 8)

In [8]:
# Паттерн 2: Конкурентное выполнение с обработкой по готовности
# Запуск всех задач сразу
tasks = [async_func(item) for item in items]

# Обработка по мере завершения
for task in asyncio.as_completed(tasks):
    result = await task
    process(result)

NameError: name 'items' is not defined

In [9]:
# Паттерн 3: Использование цикла событий внутри сопрограмм
async def io_operation():
    loop = asyncio.get_running_loop()
    # Использование loop-методов для асинхронных I/O
    await loop.sock_recv(...)

In [10]:
# Паттерн 4: Обработка исключений в сопрограммах
async def safe_probe(domain):
    try:
        return await probe(domain)
    except Exception as e:
        return (domain, False, str(e))

In [11]:
# Паттерн 5: Ограничение конкурентности
# Для предотвращения "thundering herd"
semaphore = asyncio.Semaphore(10)

async def limited_probe(domain):
    async with semaphore:
        return await probe(domain)

NameError: name 'asyncio' is not defined